# Rounded bowtie waveguide

This notebook introduces NumBAT's `rounded_bowtie` structure: two symmetric tapered inclusions with rounded inner tips, rounded outer transitions, and a circular outer domain. It builds the Gmsh geometry, displays the mesh and refractive-index profile, then solves and visualizes the first two electromagnetic modes at 1550 nm.

All geometric dimensions below are in **nm**. `alpha_tip` is a half-angle in **radians**.

## Imports and notebook helpers

The notebook locates NumBAT relative to this file, so it can be started from the repository root or from the tutorial directory. The helper below displays PNG files written by NumBAT.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2



import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

repository_root = Path.cwd()
if not (repository_root / "backend").is_dir():
    repository_root = Path.cwd().parents[1]
sys.path.append(str(repository_root / "backend"))

import materials  # noqa: E402
import numbat  # noqa: E402


def display_pngs(directory, pattern="*.png"):
    """Display NumBAT plot files created in directory."""
    for filename in sorted(Path(directory).glob(pattern)):
        display(Image(filename=str(filename)))


## Geometry and materials

The gap is measured between the nearest points of the two rounded tips. `bowlen_l` is the distance from the origin to each inclusion's circular outer boundary. The simulation boundary has radius `1.3 * bowlen_l`, providing room for evanescent optical fields.

This example uses two arsenic trisulfide (`As₂S₃`) inclusions in silica. Set `one_wedge = True` to replace the left inclusion with the background material.

In [2]:
lambda_nm = 1550
domain_x = 3000
domain_y = 3000

rad_tip = 20
alpha_tip = np.deg2rad(20)
gap_g = 30
bowlen_l = 220
rad_corner = 20
rad_wg = 500

num_modes_EM = 20
mode_indices = [0, 1]
prefix = "tt_20_rounded_bowtie"

one_wedge = False
silica = materials.make_material("SiO2_2023_Steel")
chalcogenide = materials.make_material("As2S3_2023_Steel")
material_bkg = silica
material_a = chalcogenide
material_b = material_bkg if one_wedge else material_a

## Build the rounded bowtie structure

Creating the structure loads the built-in Gmsh template, substitutes the parameters above, checks the geometry constraints, invokes Gmsh, and assigns the physical surfaces to the three selected materials.

In [3]:
nbapp = numbat.NumBATApp(prefix)
wguide = nbapp.make_structure(
    "rounded_bowtie",
    domain_x,
    domain_y,
    rad_tip=rad_tip,
    rad_wg=rad_wg,
    alpha_tip=alpha_tip,
    gap_g=gap_g,
    bowlen_l=bowlen_l,
    rad_corner=rad_corner,
    material_bkg=material_bkg,
    material_a=material_a,
    material_b=material_b,
    lc_bkg=0.02,
    lc_refine_1=5.0,
    lc_refine_2=5.0,
)

Loading plot preferences from numbat.toml.
 Calling Gmsh: 

## Gmsh mesh and refractive-index profile

The first call produces the Gmsh mesh plot. The refractive-index plotter then generates a 2D material profile and a horizontal `y = 0` cut through the bowtie tips.

In [4]:
wireframe_path, mesh_path, combined_path = wguide.plot_mesh(prefix, keep_singles=True)
print('wireframe:', wireframe_path)
print('mesh:', mesh_path)
print('combined:', combined_path)


wireframe: c:\Users\mq98002769\numbat\nb_releases\nb_latest\examples\tutorials\tt_20_rounded_bowtie-wireframe.png
mesh: c:\Users\mq98002769\numbat\nb_releases\nb_latest\examples\tutorials\tt_20_rounded_bowtie-mesh_nodes.png
combined: tt_20_rounded_bowtie-mesh.png


In [ ]:
index_plotter = wguide.get_structure_plotter_refractive_index()
index_plotter.make_plot_2D(prefix)
index_plotter.make_plot_1D_cut(prefix, "y", 0)

wireframe_path, mesh_path, combined_path = wguide.plot_mesh(prefix)
print(wireframe_path, mesh_path, combined_path)
display(Image(filename=wireframe_path))
display(Image(filename=mesh_path))
display_pngs(nbapp.outdir(), f"{prefix}*.png")

Fontconfig warning: using without calling FcInit()
Fontconfig warning: using without calling FcInit()


    initial mesh has: 6017 nodes, 2980 elements. 


 The final EM sim mesh has 6017 nodes, 2980 elements and 3 element types (materials).
 There are 3 active materials:
  SiO2_2023_Steel,     n = 1.45000+0.00000j, mat. index = 1.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 2.
  As2S3_2023_Steel,    n = 2.45000+0.00000j, mat. index = 3.
plotting to file: jup_20_rounded_bowtie-refractive_index.png
plotting to file: jup_20_rounded_bowtie-refractive_index_xcut.png


Fontconfig warning: using without calling FcInit()
Fontconfig warning: using without calling FcInit()


/home/msteel/research/numbat/devel_0_work/examples/tutorials/jup_20_rounded_bowtie-wireframe.png /home/msteel/research/numbat/devel_0_work/examples/tutorials/jup_20_rounded_bowtie-mesh_nodes.png jup_20_rounded_bowtie-mesh.png


FileNotFoundError: [Errno 2] No such file or directory: '/home/msteel/research/numbat/devel_0_work/examples/tutorials/jup_20_rounded_bowtie-wireframe.png'

## Solve electromagnetic modes

NumBAT uses an effective-index estimate to target the guided solutions. The first two returned modes are plotted below; mode `0` normally has the highest effective index.

In [ ]:
n_eff_estimate = material_a.refindex_n - 0.1
sim_EM = wguide.calc_EM_modes(num_modes_EM, lambda_nm, n_eff=n_eff_estimate)

for mode_index, n_eff in enumerate(sim_EM.neff_all()):
    print(f"mode {mode_index:2d}: n_eff = {n_eff:.6f}")

## First two 2D electromagnetic modes

These plots show the electric-field components and magnitude for modes `0` and `1`. The wire-frame border is the same rounded-bowtie geometry used by Gmsh.

In [ ]:
sim_EM.plot_modes(mode_indices=mode_indices, hide_vector_field=True)
display_pngs(nbapp.outdir_fields_path(prefix), "E*_mode_0[01].png")

## Horizontal field cuts at y = 0

The horizontal cut passes through the two rounded tips and the central gap. Comparing the first two modes along this line makes their symmetry and gap confinement easy to see.

In [ ]:
sim_EM.plot_modes_1D("y", 0, mode_indices=mode_indices)
display_pngs(nbapp.outdir_fields_path(prefix), "E*_mode_0[01]_ycut*.png")

## Next steps

NumBAT stores field plots under `jup_20_rounded_bowtie-fields/` and structure plots in the notebook's output directory. Change the geometry cell to study the tip radius, gap, half-angle, inclusion radius, or outer-corner radius. Decrease `lc_bkg` or increase the refinement factors for a finer mesh, then repeat the structure and EM-mode cells.